$\mathcal{G}:$
```
          U1     U2       U3
        ↙    ↘↙    ↘  ↙    ↘
       W ───→ Z ───→ X ────→ Y 
       ↑      ↑      ↑      
       │      U5     │
       │      ↓      │      
       U4────→M──────
```

W->Z, Z->X, X->Y, M->X

U1->W, U1->Z, U2->Z, U2->X, U3->X, U3->Y, U4->W, U4->M, U5->M, U5->Z

In [19]:
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from autobound.causalProblem import causalProblem
from autobound.DAG import DAG
from autobound.Query import Query

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


We have $\theta=(Y=1|do(X=1))$.

The mission of this notebook:
Calculate 
1. $pot(W,Z,X|do(M=1))$
2. $pot(M,Z,X|do(W=1))$


## Step 0: Simulate SCM

In [20]:
N_samples = 1000000
np.random.seed(42)

# We are simulating a ground truth SCM. That is, we will 
# 1. randomly pick all root variables, 
# 2. sample other variables as some arbitrary boolean function of their parents.
# We will simulate the actual confounders (not response types!) as binary variables.

# 1. Simulate Root variables (all unobserved confounders)
U1 = np.random.binomial(1, 0.4, N_samples)
U2 = np.random.binomial(1, 0.6, N_samples)
U3 = np.random.binomial(1, 0.3, N_samples)
U4 = np.random.binomial(1, 0.5, N_samples)
U5 = np.random.binomial(1, 0.7, N_samples)

# 2. Structural equations
W = U1 ^ U4
M = U4 & U5
Z = (W & U2) ^ (U1 | U5)
X = (Z ^ U2) | (M & U3)
Y = X ^ U3

ground_truth_data = pd.DataFrame({
    'W': W,
    'Z': Z,
    'X': X,
    'Y': Y,
    'M': M,
    'U1': U1,
    'U2': U2,
    'U3': U3,
    'U4': U4,
    'U5': U5
})

## Step 1: Compute $\mathcal{W}_\emptyset$

In [21]:
# Observational data
obs = ground_truth_data.drop(columns=['U1', 'U2', 'U3', 'U4', 'U5'])
obs_counts = obs.value_counts().reset_index(name='count')

# Round probabilities and force exact normalization after rounding
obs_counts['prob'] = (obs_counts['count'] / N_samples).round(6)
diff = round(1.0 - obs_counts['prob'].sum(), 6)

if diff != 0:
    idx = obs_counts['count'].idxmax()
    obs_counts.loc[idx, 'prob'] = round(obs_counts.loc[idx, 'prob'] + diff, 6)

obs_counts.drop(columns=['count']).to_csv('data/figure-15-obs.csv', index=False)

In [22]:
obs_counts.drop(columns=['count'])

,W,Z,X,Y,M,prob
0,0,1,0,0,0,0.113389
1,1,0,1,1,1,0.088216
2,1,0,1,1,0,0.084413
3,0,1,1,1,0,0.075112
4,0,1,0,0,1,0.058872
5,1,1,1,1,1,0.058578
6,1,1,1,1,0,0.056188
7,0,1,0,1,0,0.048643
8,0,1,1,0,1,0.042096
9,0,1,1,1,1,0.039085


In [23]:
dag = DAG()
dag.from_structure("W->Z, Z->X, X->Y, M->X, " \
"U1->W, U1->Z, U2->Z, U2->X, U3->X, U3->Y, U4->W, U4->M, U5->M, U5->Z", unob='U1,U2,U3,U4,U5')
problem = causalProblem(dag)
problem.load_data('data/figure-15-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1)=1'))
program = problem.write_program()

(lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
print(f"P(θ)∈[{lb:.4f}, {ub:.4f}]")
calW_empty = ub - lb
print(f"calW_empty = {calW_empty:.8f}")

P(θ)∈[0.4393, 0.7863]
calW_empty = 0.34700000


## Step 2: Compute $\mathcal{W_{a}}$

### Algorithm: Approximating $W_a(p_a)$ via Grid Search

We want to compute the worst-case width $W_a = \max_{p_a} W(p_a)$ of the identified set
for $\theta = P(Y=1 | do(X=1))$, maximized over all possible experimental outcomes $p_a$.

Since we don't know the true interventional distribution $p_a$, we search over all
candidate probability vectors that could plausibly arise from the experiment.

1. **Bound the search space** using observational data (Tian & Pearl, 2000, eq. 22):
   For each outcome tuple, the interventional probability is bounded by:
     - Lower bound: $P(\text{interv\_var}=v, \text{outcome})$
     - Upper bound: $1 - P(\text{interv\_var}=v) + P(\text{interv\_var}=v, \text{outcome})$

2. **Discretize** each bounded interval into a grid with step size $1/n$.

3. **Enumerate** all grid vectors that sum to 1 and fall within the bounds.

4. **For each candidate vector**, solve the partial identification problem and record the width.

5. **Return the maximum width** across all feasible candidate vectors.
   The potency is then $pot(a) = W_\emptyset - W_a$.

### Computing true interventions.
Let's try the ground-truth $p$ first. If $W_a(p) = W_\emptyset$, we don't need to loop over all other feasible $p$.

In [24]:
def getIntervention_doM(M_val):
    X_doM = (Z ^ U2) | (M_val & U3)
    doM = pd.DataFrame({'W_doM': W, 'Z_doM': Z, 'X_doM': X_doM})
    doM_counts = doM.value_counts().reset_index(name='count')
    doM_counts['prob'] = (doM_counts['count'] / N_samples).round(6)
    return doM_counts.drop(columns=['count'])

def getIntervention_doW(W_val):
    Z_doW = (W_val & U2) ^ (U1 | U5)
    X_doW = (Z_doW ^ U2) | (M & U3)
    doW = pd.DataFrame({'M_doW': M, 'Z_doW': Z_doW, 'X_doW': X_doW})
    doW_counts = doW.value_counts().reset_index(name='count')
    doW_counts['prob'] = (doW_counts['count'] / N_samples).round(6)
    return doW_counts.drop(columns=['count'])

print("True P(W,Z,X | do(M=1)):")
print(getIntervention_doM(1))
print("\nTrue P(M,Z,X | do(W=1)):")
print(getIntervention_doW(1))

True P(W,Z,X | do(M=1)):
   W_doM  Z_doM  X_doM      prob
0      1      0      1  0.257206
1      0      1      1  0.237246
2      1      1      1  0.180037
3      0      1      0  0.172261
4      0      0      1  0.064822
5      1      1      0  0.038056
6      1      0      0  0.025213
7      0      0      0  0.025159

True P(M,Z,X | do(W=1)):
   M_doW  Z_doW  X_doW      prob
0      0      0      1  0.282465
1      1      0      1  0.210425
2      0      1      1  0.187628
3      1      1      1  0.139418
4      0      1      0  0.108167
5      0      0      0  0.071897


### $pot(W,Z,X | do(M=1))$

In [26]:
def W_doM(p_config_0, p_config_1):
    problem = causalProblem(dag)
    problem.load_data('data/figure-15-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1)=1'))

    for M_val, p_config in [(0, p_config_0), (1, p_config_1)]:
        for _, row in p_config.iterrows():
            w, z, x = int(row.W_doM), int(row.Z_doM), int(row.X_doM)
            lhs = problem.query(f'W(M={M_val})={w}&Z(M={M_val})={z}&X(M={M_val})={x}')
            problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb

In [27]:
true_doM0 = getIntervention_doM(0)
true_doM1 = getIntervention_doM(1)
print("True P(W,Z,X | do(M=0)):")
print(true_doM0)
print("\nTrue P(W,Z,X | do(M=1)):")
print(true_doM1)

W_doM_true = W_doM(true_doM0, true_doM1)
print(f"\nW_doM with true p = {W_doM_true:.8f}")
print(f"pot(W,Z,X|do(M))  = {calW_empty - W_doM_true:.8f}")

True P(W,Z,X | do(M=0)):
   W_doM  Z_doM  X_doM      prob
0      1      0      1  0.246535
1      0      1      0  0.246355
2      1      1      1  0.163894
3      0      1      1  0.163152
4      1      1      0  0.054199
5      0      0      1  0.053968
6      0      0      0  0.036013
7      1      0      0  0.035884

True P(W,Z,X | do(M=1)):
   W_doM  Z_doM  X_doM      prob
0      1      0      1  0.257206
1      0      1      1  0.237246
2      1      1      1  0.180037
3      0      1      0  0.172261
4      0      0      1  0.064822
5      1      1      0  0.038056
6      1      0      0  0.025213
7      0      0      0  0.025159

W_doM with true p = 0.34700230
pot(W,Z,X|do(M))  = -0.00000230


### $pot(M,Z,X | do(W=1))$

In [28]:
def W_doW(p_config_0, p_config_1):
    problem = causalProblem(dag)
    problem.load_data('data/figure-15-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1)=1'))

    for W_val, p_config in [(0, p_config_0), (1, p_config_1)]:
        for _, row in p_config.iterrows():
            m, z, x = int(row.M_doW), int(row.Z_doW), int(row.X_doW)
            lhs = problem.query(f'M(W={W_val})={m}&Z(W={W_val})={z}&X(W={W_val})={x}')
            problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb

In [29]:
true_doW0 = getIntervention_doW(0)
true_doW1 = getIntervention_doW(1)
print("True P(M,Z,X | do(W=0)):")
print(true_doW0)
print("\nTrue P(M,Z,X | do(W=1)):")
print(true_doW1)

W_doW_true = W_doW(true_doW0, true_doW1)
print(f"\nW_doW with true p = {W_doW_true:.8f}")
print(f"pot(M,Z,X|do(W))  = {calW_empty - W_doW_true:.8f}")

True P(M,Z,X | do(W=0)):
   M_doW  Z_doW  X_doW      prob
0      0      1      0  0.282465
1      1      1      1  0.202755
2      0      1      1  0.187628
3      1      1      0  0.147088
4      0      0      1  0.108167
5      0      0      0  0.071897

True P(M,Z,X | do(W=1)):
   M_doW  Z_doW  X_doW      prob
0      0      0      1  0.282465
1      1      0      1  0.210425
2      0      1      1  0.187628
3      1      1      1  0.139418
4      0      1      0  0.108167
5      0      0      0  0.071897

W_doW with true p = 0.34700235
pot(M,Z,X|do(W))  = -0.00000235


In [30]:
print(f"W_empty = {calW_empty:.8f}")
print(f"W_doM   = {W_doM_true:.8f}, W_doW   = {W_doW_true:.8f}")
print(f"pot(W,Z,X|do(M))={calW_empty - W_doM_true:.8f}")
print(f"pot(M,Z,X|do(W))={calW_empty - W_doW_true:.8f}")

W_empty = 0.34700000
W_doM   = 0.34700230, W_doW   = 0.34700235
pot(W,Z,X|do(M))=-0.00000230
pot(M,Z,X|do(W))=-0.00000235


$\exists p^* : W_a(p^*) = W_\emptyset \implies pot(a)=0$

## Bonus: Grid search over feasible $p$
We already found a $p^*$ that leads to $pot=0$. For completeness, let's still loop over all feasible $p$ vectors and take the maximum.

In [25]:
from itertools import product

def generate_feasible_vectors(n, bounds_list):
    """Generate all probability vectors on grid 1/n, summing to 1, within bounds."""
    k = len(bounds_list)

    def recurse(idx, remaining, current):
        if idx == k - 1:
            val = remaining / n
            lb, ub = bounds_list[idx]
            if lb - 1e-9 <= val <= ub + 1e-9:
                yield current + [val]
            return
        lb, ub = bounds_list[idx]
        lo = max(int(np.ceil(lb * n)), 0)
        hi = min(int(np.floor(ub * n)), remaining)
        for a in range(lo, hi + 1):
            yield from recurse(idx + 1, remaining - a, current + [a / n])

    return list(recurse(0, n, []))


def sample_feasible_vectors(n_samples, bounds_list, seed=42):
    """Sample n_samples probability vectors from the feasible polytope.
    
    Feasible = within component bounds and sums to 1.
    Strategy: sample deltas ~ Uniform(0, slack_i), rescale to fill
    the remaining mass (1 - sum(lb)), reject if any component exceeds its bound.
    """
    lbs = np.array([b[0] for b in bounds_list])
    ubs = np.array([b[1] for b in bounds_list])
    slacks = ubs - lbs
    remaining = 1.0 - lbs.sum()

    rng = np.random.default_rng(seed)
    samples = []
    attempts = 0
    while len(samples) < n_samples and attempts < n_samples * 10000:
        attempts += 1
        deltas = rng.uniform(0, slacks)
        s = deltas.sum()
        if s < 1e-12:
            continue
        deltas = deltas * (remaining / s)
        if np.all(deltas <= slacks + 1e-9):
            samples.append((lbs + deltas).tolist())

    print(f"  Sampled {len(samples)} vectors ({attempts} attempts, "
          f"{attempts / max(len(samples), 1):.0f}x rejection ratio)")
    return samples

In [31]:
def compute_bounds(interv_var, interv_val, outcome_vars, obs):
    """Compute eq. 22 bounds on P(outcome | do(interv_var=val)) from obs data.
    
    For each outcome tuple: lb = P(interv=v, outcome), ub = 1 - P(interv=v) + lb
    """
    group_vars = [interv_var] + outcome_vars
    obs_marg = obs.groupby(group_vars)['prob'].sum().reset_index()
    p_interv = obs_marg[obs_marg[interv_var] == interv_val]['prob'].sum()
    
    outcomes = list(product([0, 1], repeat=len(outcome_vars)))
    bounds = []
    for vals in outcomes:
        mask = obs_marg[interv_var] == interv_val
        for var, val in zip(outcome_vars, vals):
            mask &= obs_marg[var] == val
        p_joint = obs_marg.loc[mask, 'prob'].sum()
        bounds.append((p_joint, 1 - p_interv + p_joint))
    return outcomes, bounds


def make_config_df(vec, outcomes, prefix):
    """Build a DataFrame from a probability vector and outcome tuples."""
    cols = [f'{v}_{prefix}' for v in ['placeholder'] * len(outcomes[0])]
    data = {f'{col}': [o[i] for o in outcomes] for i, col in enumerate(cols)}
    data['prob'] = vec
    return pd.DataFrame(data)

obs = obs_counts.drop(columns=['count'])

In [32]:
# Grid search for do(M): sample feasible (p_doM0, p_doM1) pairs
N_CALLS = 100  # target number of solver calls
n_per = int(np.ceil(np.sqrt(N_CALLS)))  # samples per intervention value

outcome_vars_M = ['W', 'Z', 'X']
outcomes_M, bounds_M0 = compute_bounds('M', 0, outcome_vars_M, obs)
_, bounds_M1 = compute_bounds('M', 1, outcome_vars_M, obs)

print(f"Bounds for do(M=0): {[(f'{lb:.3f}', f'{ub:.3f}') for lb, ub in bounds_M0]}")
print(f"Bounds for do(M=1): {[(f'{lb:.3f}', f'{ub:.3f}') for lb, ub in bounds_M1]}")

print(f"\nSampling ~{n_per} vectors per intervention value:")
vecs_M0 = sample_feasible_vectors(n_per, bounds_M0, seed=42)
vecs_M1 = sample_feasible_vectors(n_per, bounds_M1, seed=43)

total = len(vecs_M0) * len(vecs_M1)
print(f"Total solver calls: {total}")

W_doM_grid = []
for i, v0 in enumerate(vecs_M0):
    for j, v1 in enumerate(vecs_M1):
        df0 = pd.DataFrame({
            'W_doM': [o[0] for o in outcomes_M], 'Z_doM': [o[1] for o in outcomes_M],
            'X_doM': [o[2] for o in outcomes_M], 'prob': v0
        })
        df1 = pd.DataFrame({
            'W_doM': [o[0] for o in outcomes_M], 'Z_doM': [o[1] for o in outcomes_M],
            'X_doM': [o[2] for o in outcomes_M], 'prob': v1
        })
        try:
            w = W_doM(df0, df1)
            W_doM_grid.append(w)
        except Exception as e:
            continue
        idx = i * len(vecs_M1) + j + 1
        if idx % 10 == 0:
            print(f"  [{idx}/{total}] current max W = {max(W_doM_grid):.6f}")

calW_doM_grid = max(W_doM_grid)
print(f"\ncalW_doM (sampled) = {calW_doM_grid:.8f}")
print(f"pot(W,Z,X|do(M))   = {calW_empty - calW_doM_grid:.8f}")

Bounds for do(M=0): [('0.036', '0.386'), ('0.054', '0.404'), ('0.162', '0.512'), ('0.107', '0.457'), ('0.036', '0.386'), ('0.120', '0.470'), ('0.054', '0.404'), ('0.080', '0.430')]
Bounds for do(M=1): [('0.000', '0.650'), ('0.000', '0.650'), ('0.059', '0.709'), ('0.081', '0.731'), ('0.000', '0.650'), ('0.126', '0.776'), ('0.000', '0.650'), ('0.084', '0.734')]

Sampling ~10 vectors per intervention value:
  Sampled 10 vectors (10 attempts, 1x rejection ratio)
  Sampled 10 vectors (10 attempts, 1x rejection ratio)
Total solver calls: 100
  [10/100] current max W = 0.359561
  [20/100] current max W = 0.359561
  [30/100] current max W = 0.359561
  [40/100] current max W = 0.359561
  [50/100] current max W = 0.359561
  [60/100] current max W = 0.359561
  [70/100] current max W = 0.359561
  [80/100] current max W = 0.359561
  [90/100] current max W = 0.359561
  [100/100] current max W = 0.359561

calW_doM (sampled) = 0.35956110
pot(W,Z,X|do(M))   = -0.01256110


In [33]:
# Grid search for do(W): sample feasible (p_doW0, p_doW1) pairs
outcome_vars_W = ['M', 'Z', 'X']
outcomes_W, bounds_W0 = compute_bounds('W', 0, outcome_vars_W, obs)
_, bounds_W1 = compute_bounds('W', 1, outcome_vars_W, obs)

print(f"Bounds for do(W=0): {[(f'{lb:.3f}', f'{ub:.3f}') for lb, ub in bounds_W0]}")
print(f"Bounds for do(W=1): {[(f'{lb:.3f}', f'{ub:.3f}') for lb, ub in bounds_W1]}")

print(f"\nSampling ~{n_per} vectors per intervention value:")
vecs_W0 = sample_feasible_vectors(n_per, bounds_W0, seed=44)
vecs_W1 = sample_feasible_vectors(n_per, bounds_W1, seed=45)

total = len(vecs_W0) * len(vecs_W1)
print(f"Total solver calls: {total}")

W_doW_grid = []
for i, v0 in enumerate(vecs_W0):
    for j, v1 in enumerate(vecs_W1):
        df0 = pd.DataFrame({
            'M_doW': [o[0] for o in outcomes_W], 'Z_doW': [o[1] for o in outcomes_W],
            'X_doW': [o[2] for o in outcomes_W], 'prob': v0
        })
        df1 = pd.DataFrame({
            'M_doW': [o[0] for o in outcomes_W], 'Z_doW': [o[1] for o in outcomes_W],
            'X_doW': [o[2] for o in outcomes_W], 'prob': v1
        })
        try:
            w = W_doW(df0, df1)
            W_doW_grid.append(w)
        except Exception as e:
            continue
        idx = i * len(vecs_W1) + j + 1
        if idx % 10 == 0:
            print(f"  [{idx}/{total}] current max W = {max(W_doW_grid):.6f}")

calW_doW_grid = max(W_doW_grid)
print(f"\ncalW_doW (sampled) = {calW_doW_grid:.8f}")
print(f"pot(M,Z,X|do(W))   = {calW_empty - calW_doW_grid:.8f}")

Bounds for do(W=0): [('0.036', '0.537'), ('0.054', '0.554'), ('0.162', '0.663'), ('0.107', '0.608'), ('0.000', '0.501'), ('0.000', '0.501'), ('0.059', '0.559'), ('0.081', '0.582')]
Bounds for do(W=1): [('0.036', '0.535'), ('0.120', '0.620'), ('0.054', '0.554'), ('0.080', '0.580'), ('0.000', '0.499'), ('0.126', '0.626'), ('0.000', '0.499'), ('0.084', '0.583')]

Sampling ~10 vectors per intervention value:
  Sampled 10 vectors (10 attempts, 1x rejection ratio)
  Sampled 10 vectors (10 attempts, 1x rejection ratio)
Total solver calls: 100
  [10/100] current max W = 0.289773
  [20/100] current max W = 0.289773
  [30/100] current max W = 0.289773
  [40/100] current max W = 0.301055
  [50/100] current max W = 0.352008
  [60/100] current max W = 0.352008
  [70/100] current max W = 0.352008
  [80/100] current max W = 0.352008
  [90/100] current max W = 0.352008
  [100/100] current max W = 0.352008

calW_doW (sampled) = 0.35200824
pot(M,Z,X|do(W))   = -0.00500824


Result: potency is zero down to machine precision